# 05 · Validação dos dados e centralidades — rede de colaboração UNIFEI

Calcula os números citados na Seção 3 do artigo que não saem dos outros notebooks:

1. **Centralidades** (grau, proximidade, intermediação), concentração de conexões (Gini, top 1% e 10%), pontos de articulação e arestas-ponte do maior componente (Seção 3.8.2).
2. **Nomes duplicados** (mesmo nome com mais de um identificador do OpenAlex) e **análise de sensibilidade**: efeito de fundir esses identificadores sobre as métricas da rede (Tabela de qualidade).
3. **Consistência da atribuição de instituto**: fração de vínculos em que o nome do instituto aparece na mesma afiliação que cita a UNIFEI, Itajubá ou Itabira.

**Entradas** (em `DATA_DIR`): `nodes.csv`, `edges.csv`, `pesquisadores_unifei_todos_institutos.csv`.
**Observações:** a fusão por nome é um *limite superior* da fragmentação, pois também junta homônimos. A checagem 3 mede consistência interna da regra; não substitui validação contra fonte externa.

In [ ]:
import os, re, html, unicodedata
import numpy as np
import pandas as pd
import networkx as nx

try:                                   # Colab
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = os.environ.get("DATA_DIR", "/content/drive/MyDrive")
except ImportError:                    # execução local
    DATA_DIR = os.environ.get("DATA_DIR", "data")

nodes = pd.read_csv(os.path.join(DATA_DIR, "nodes.csv"))
edges = pd.read_csv(os.path.join(DATA_DIR, "edges.csv"))
pesq  = pd.read_csv(os.path.join(DATA_DIR, "pesquisadores_unifei_todos_institutos.csv"))

G = nx.Graph()
G.add_nodes_from(nodes["id"])
for s, t, w in zip(edges["source"], edges["target"], edges["weight"]):
    G.add_edge(s, t, weight=w)

Gc = G.subgraph(max(nx.connected_components(G), key=len)).copy()
rotulo = dict(zip(nodes["id"], nodes["label"]))
instit = dict(zip(nodes["id"], nodes["instituto"]))
print(f"Rede geral: {G.number_of_nodes()} nós, {G.number_of_edges()} arestas, "
      f"{nx.number_connected_components(G)} componentes")
print(f"Maior componente: {Gc.number_of_nodes()} nós, {Gc.number_of_edges()} arestas")

## 1. Centralidades e concentração (maior componente)

Grau, proximidade (*closeness*) e intermediação (*betweenness*, exata, sem normalizar). A intermediação leva cerca de 1 minuto.

In [ ]:
grau = dict(Gc.degree())
prox = nx.closeness_centrality(Gc)
inter = nx.betweenness_centrality(Gc, normalized=False)

def topo(d, k=5):
    return [(rotulo[i], instit[i], d[i]) for i in sorted(d, key=d.get, reverse=True)[:k]]

print("Top 5 por grau:");        [print("  ", x) for x in topo(grau)]
print("Top 5 por proximidade:"); [print("  ", (a, b, round(c, 3))) for a, b, c in topo(prox)]
print("Top 5 por intermediação (x10^3):"); [print("  ", (a, b, round(c / 1000))) for a, b, c in topo(inter)]

def top10(d): return set(sorted(d, key=d.get, reverse=True)[:10])
print("\nSobreposição entre os 10 primeiros:",
      "grau∩proximidade =", len(top10(grau) & top10(prox)),
      "| grau∩intermediação =", len(top10(grau) & top10(inter)),
      "| proximidade∩intermediação =", len(top10(prox) & top10(inter)))

# Concentração de conexões
g = np.array(sorted(grau.values(), reverse=True)); n = len(g)
top1, top10p = int(n * 0.01), int(n * 0.10)
print(f"\nTop 1% ({top1} nós) detém {g[:top1].sum() / g.sum() * 100:.1f}% da soma dos graus")
print(f"Top 10% ({top10p} nós) detém {g[:top10p].sum() / g.sum() * 100:.1f}% da soma dos graus")
x = np.sort(g); gini = (2 * np.sum(np.arange(1, n + 1) * x) / (n * x.sum())) - (n + 1) / n
print(f"Gini do grau: {gini:.3f}")

# Pontos de articulação e arestas-ponte
print(f"\nPontos de articulação: {len(list(nx.articulation_points(Gc)))}")
print(f"Arestas-ponte:         {len(list(nx.bridges(Gc)))}")

## 2. Nomes duplicados e sensibilidade à fragmentação de autores

Um mesmo pesquisador pode aparecer com mais de um identificador no OpenAlex. Aqui contamos nomes repetidos (normalizados: sem acento, caixa ou pontuação) e refazemos a rede fundindo os identificadores de mesmo nome com **3 ou mais palavras** (para reduzir o risco de juntar homônimos curtos).

In [ ]:
def norm(s):
    s = unicodedata.normalize("NFKD", html.unescape(str(s))).encode("ascii", "ignore").decode().lower()
    return re.sub(r"[^a-z ]", "", s).strip()

nodes["nome_norm"] = nodes["label"].map(norm)
dup = nodes[nodes["nome_norm"].duplicated(keep=False)]
print(f"Nomes com mais de um id: {dup['nome_norm'].nunique()} nomes -> {len(dup)} ids")

def metricas(Gx, rotulo_):
    cc = max(nx.connected_components(Gx), key=len); Gm = Gx.subgraph(cc)
    d = [v for _, v in Gm.degree()]
    return {"nós": Gx.number_of_nodes(), "arestas": Gx.number_of_edges(),
            "componentes": nx.number_connected_components(Gx), "maior comp.": len(cc),
            "isolados": sum(1 for _ in nx.isolates(Gx)), "grau máx.": max(d),
            "grau médio": round(float(np.mean(d)), 2),
            "CA médio": round(nx.average_clustering(Gm), 3),
            "CA global": round(nx.transitivity(Gm), 3)}

# fusão: representante = id de maior grau entre os de mesmo nome (>= 3 palavras)
grau_g = dict(G.degree())
rep = {}
for nome, grp in nodes.groupby("nome_norm"):
    if len(grp) > 1 and len(nome.split()) >= 3:
        r = max(grp["id"], key=lambda i: grau_g[i])
        for i in grp["id"]:
            rep[i] = r
print(f"Ids fundidos: {sum(1 for i, r in rep.items() if i != r)} (em {len(set(rep.values()))} nomes)")

Gf = nx.Graph(); Gf.add_nodes_from({rep.get(i, i) for i in nodes["id"]})
for s, t in zip(edges["source"], edges["target"]):
    a, b = rep.get(s, s), rep.get(t, t)
    if a != b:
        Gf.add_edge(a, b)

tabela = pd.DataFrame({"original": metricas(G, "orig"), "fundido": metricas(Gf, "fund")})
tabela

## 3. Consistência da atribuição de instituto

Reaplica a regra do notebook de extração (nome completo do instituto, em minúsculas) e verifica, para cada vínculo atribuído, se o nome do instituto aparece **na mesma string de afiliação** que menciona UNIFEI, Itajubá ou Itabira. Mede consistência interna; não é validação externa.

In [ ]:
# Mesmo dicionário usado em 01_extracao (mantenha idêntico)
INSTITUTOS_UNIFEI = {
    "IEM":   ["Instituto de Engenharia Mecânica", "Institute of Mechanical Engineering"],
    "IEPG":  ["Instituto de Engenharia de Produção e Gestão", "Instituto de Produção e Gestão",
              "Institute of Production Engineering and Management",
              "Institute of Industrial Engineering and Management"],
    "IESTI": ["Instituto de Engenharia de Sistemas e Tecnologia da Informação",
              "Institute of Systems Engineering and Information Technology"],
    "IFQ":   ["Instituto de Física e Química", "Institute of Physics and Chemistry"],
    "IMC":   ["Instituto de Matemática e Computação", "Institute of Mathematics and Computing"],
    "IRN":   ["Instituto de Recursos Naturais", "Institute of Natural Resources"],
    "ISEE":  ["Instituto de Sistemas Elétricos e Energia", "Institute of Electrical Systems and Energy",
              "Institute of Electrical Systems"],
    "IEI":   ["Instituto de Engenharias Integradas", "Institute of Integrated Engineering"],
}
menciona_unifei = re.compile(r"unifei|itajub|itabira")

ok = total = 0
for _, r in pesq.iterrows():
    if r["instituto"] == "Não identificado":
        continue
    cadeias = str(r["afiliacoes"]).split(" | ")
    for sigla in [s.strip() for s in r["instituto"].split(";")]:
        total += 1
        ok += any(p.lower() in c.lower() and menciona_unifei.search(norm(c))
                  for c in cadeias for p in INSTITUTOS_UNIFEI[sigla])
print(f"Vínculos com o nome do instituto na mesma afiliação que cita a UNIFEI: {ok} de {total} ({ok / total * 100:.1f}%)")